# Run QCL-SAM on the DGX A100

Run the cells in order. This trains OpenEarthMap followed by LandCover.ai and displays the saved results. Defaults use your server layout. No training results are prefilled.

**Before opening:** update the repository successfully (see the README's blocked-pull recovery section). In the VS Code Remote SSH terminal, register the existing environment once:

```bash
cd /raid/workspace/AI4CV/AI4CV_CL_DGX_A100/QCL_src
source .venv/bin/activate
python -m pip install ipykernel
python -m ipykernel install --user --name qcl-dgx --display-name "QCL DGX (.venv)"
```

Open this notebook and choose **QCL DGX (.venv)** as its kernel. Keep the server kernel running while training. The notebook does not resume interrupted training; each setup run creates a new output directory. Batch progress appears at batch 1, every 25 batches, and the last batch. Epoch summaries include training and validation.

In [ ]:
from pathlib import Path
import os
import sys
from datetime import datetime

REPO = Path("/raid/workspace/AI4CV/AI4CV_CL_DGX_A100")
SAM_CHECKPOINT = Path("/raid/workspace/AI4CV/models/sam_vit_b_01ec64.pth")
GPU = "0"
EPOCHS = 24                  # Set to 1 for an initial end-to-end check (not final results).
BATCH_SIZE = 2
NUM_WORKERS = 4              # Set 0 if the server reports shared-memory/worker errors.
SAM_PRECISION = "bfloat16"   # A100 acceleration; "float32" is the baseline.
RESIDUAL_SCALE = 0.1         # SAM features + 0.1 * quantum features; 0 restores legacy behavior.
INSTALL_DEPENDENCIES = False # True only if this environment still needs requirements.txt.

# Leave None to detect from the two dataset locations in your screenshot.
# Set explicit paths here if detection reports missing or ambiguous directories.
OEM_ROOT = None
LANDCOVER_ROOT = None

PROJECT = REPO / "QCL_src"
assert PROJECT.is_dir(), f"Project not found: {PROJECT}"
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), (
    f"Select QCL DGX (.venv). Current Python: {sys.executable}, prefix: {sys.prefix}"
)
assert SAM_CHECKPOINT.is_file(), f"Checkpoint not found: {SAM_CHECKPOINT}"
assert EPOCHS >= 1 and BATCH_SIZE >= 1
os.chdir(PROJECT)
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
os.environ["SAM_CHECKPOINT"] = str(SAM_CHECKPOINT)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = PROJECT / "outputs" / ("dgx_notebook_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
VALIDATED = False
TRAINING_COMPLETE = False
print("Python:", sys.executable)
print("Checkpoint:", SAM_CHECKPOINT)
print("Run output:", RUN_DIR)

### Optimized run settings

Frozen SAM uses BF16 on the A100; the quantum circuit and trainable head stay in FP32. Metrics remain on the prediction device. Four data workers prefetch images. SAM is still frozen and is not cached.

The residual path changes the architecture: the head sees SAM's spatial features plus the quantum adaptation, instead of only the coarse quantum output. This is intended to preserve spatial information, not a guarantee of higher accuracy. Set RESIDUAL_SCALE=0 for a baseline comparison. Use each run's saved config when evaluating its checkpoints. Compare per-class IoU and validation metrics, not accuracy alone.


## Environment and GPU

Dependency installation is optional. GPU validation runs in a fresh process, so the selected GPU is respected even if the notebook kernel previously imported PyTorch. All training and validation commands use this kernel's Python and the current checkout.

In [ ]:
import subprocess
import json

if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

# Run CPU regressions and GPU checks (GPU-specific tests skip on CPU-only hosts).
tests = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_dgx_optimization.py", "-v"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(tests.stdout)
tests.check_returncode()

probe = """
import torch, yaml, pennylane, segment_anything, sklearn, matplotlib
import qcl_sam_seg
print("Package:", qcl_sam_seg.__file__)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA unavailable: check the selected GPU and the server PyTorch installation"
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
"""
for command in ([sys.executable, "-u", "-c", probe], ["nvidia-smi"]):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout)
    result.check_returncode()
revision = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
(RUN_DIR / "environment.json").write_text(json.dumps({
    "git_commit": revision, "python": sys.executable, "gpu": GPU,
    "sam_checkpoint": str(SAM_CHECKPOINT), "epochs": EPOCHS, "batch_size": BATCH_SIZE,
}, indent=2))
print("Git commit:", revision)

## Create run-specific configs

Tracked YAML files are preserved. Copies contain absolute DGX paths, the explicit SAM checkpoint, and this run's output directory. Existing split options (including the OpenEarthMap availability filter and holdout) are preserved. If both candidate dataset locations exist, set the desired root in the first cell and rerun setup.

In [ ]:
import copy
import yaml

def choose_root(explicit, relative):
    if explicit is not None:
        path = Path(explicit).expanduser().resolve()
        assert path.is_dir(), f"Dataset directory missing: {path}"
        return path
    candidates = list(dict.fromkeys(
        p.resolve() for p in (REPO / "A100_datasets" / relative,
                              REPO.parent / "A100_datasets" / relative) if p.is_dir()
    ))
    if len(candidates) != 1:
        raise ValueError(f"Set an explicit dataset root for {relative}; found: {candidates}")
    return candidates[0]

roots = {
    "openearthmap": choose_root(OEM_ROOT, Path("openearthmap") / "OpenEarthMap_wo_xBD"),
    "landcoverai": choose_root(LANDCOVER_ROOT, Path("landcover_ai")),
}
CONFIG_DIR = RUN_DIR / "configs"
CONFIG_DIR.mkdir(exist_ok=True)
CONFIGS = {}
for name, root in roots.items():
    source = PROJECT / "configs" / "datasets" / f"{name}.yaml"
    cfg = copy.deepcopy(yaml.safe_load(source.read_text()))
    old_root = (source.parent / cfg["dataset"]["root"]).resolve()
    cfg["dataset"]["root"] = str(root)
    for split, spec in cfg["dataset"]["splits"].items():
        old_value = spec if isinstance(spec, str) else spec["manifest"]
        old_manifest = (source.parent / old_value).resolve()
        try:
            manifest = root / old_manifest.relative_to(old_root)
        except ValueError:
            # A custom manifest outside the dataset retains its original absolute path.
            manifest = old_manifest
        assert manifest.is_file(), f"{name}/{split} manifest not found: {manifest}"
        if isinstance(spec, str):
            cfg["dataset"]["splits"][split] = str(manifest)
        else:
            spec["manifest"] = str(manifest)
    cfg["model"]["sam_checkpoint"] = str(SAM_CHECKPOINT)
    cfg["model"].update(sam_precision=SAM_PRECISION, residual_scale=RESIDUAL_SCALE)
    cfg["training"].update(epochs=EPOCHS, batch_size=BATCH_SIZE,
                           num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)
    cfg.setdefault("output", {})["root"] = str(RUN_DIR)
    destination = CONFIG_DIR / f"{name}.yaml"
    destination.write_text(yaml.safe_dump(cfg, sort_keys=False))
    CONFIGS[name] = destination
    print(name, "->", root)

STREAM_NAME = "openearthmap_then_landcoverai"
STREAM = CONFIG_DIR / "stream.yaml"
STREAM.write_text(yaml.safe_dump({
    "name": STREAM_NAME, "tasks": [str(p) for p in CONFIGS.values()]
}, sort_keys=False))
RESULTS = RUN_DIR / STREAM_NAME
print("Stream:", STREAM)

## Validate the datasets

Checks that train/validation/test splits have usable pairs and no shared sample IDs, and scans the configured number of masks for unknown labels. A validation error stops this cell; fix the reported paths or data before training.

In [ ]:
import threading
import queue

def run_logged(arguments, log_path):
    """Stream child output, save it, and propagate failures or interrupts."""
    command = [sys.executable, "-u", "-m", "qcl_sam_seg", *map(str, arguments)]
    print("Running:", " ".join(command), flush=True)
    messages = queue.Queue()
    with Path(log_path).open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command, cwd=PROJECT, env=os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        def reader():
            try:
                for line in process.stdout:
                    messages.put(line)
            finally:
                messages.put(None)
        thread = threading.Thread(target=reader, daemon=True)
        thread.start()
        try:
            while True:
                try:
                    line = messages.get(timeout=0.5)
                except queue.Empty:
                    continue
                if line is None:
                    break
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
            if code:
                raise subprocess.CalledProcessError(code, command)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
        finally:
            thread.join(timeout=2)
            process.stdout.close()
    print("Saved log:", log_path)

VALIDATED = False
for name, config in CONFIGS.items():
    run_logged(["prepare-data", "--config", config], RUN_DIR / f"validate_{name}.log")
VALIDATED = True
print("Both datasets passed validation.")

## Train and evaluate

This cell runs both tasks on the selected GPU. The package automatically evaluates after each task and saves predictions, t-SNE plots, metrics, and checkpoints. The quantum circuit executes on CPU while SAM and the semantic heads use the GPU.

This is a full training run, not a resume command. To start another run, rerun from the first setup cell. Interrupting this cell terminates its training subprocess. Errors are retained in the log and raised in the notebook.

In [ ]:
assert VALIDATED, "Run the dataset validation cell successfully first."
assert not RESULTS.exists(), "This run already started. Rerun setup for a new output directory."
TRAINING_COMPLETE = False
run_logged(["train", "--stream", STREAM], RUN_DIR / "dgx_training.log")
expected = [RESULTS / name / "latest.pt" for name in CONFIGS]
assert all(path.is_file() for path in expected), f"Expected checkpoints: {expected}"
TRAINING_COMPLETE = True
print("Training and evaluation complete.")
print("Final continual checkpoint:", RESULTS / "landcoverai" / "latest.pt")

## Metrics and training curves

Per-task test metrics are measured at the end of that task's training. The final LandCover.ai stage's forgetting report contains both tasks evaluated after the second task. View existing results by setting `RESULTS` to an earlier run's stream directory below; training need not be rerun.

In [ ]:
# To inspect an earlier run:
# RESULTS = Path("/raid/workspace/AI4CV/AI4CV_CL_DGX_A100/QCL_src/outputs/dgx_notebook_.../openearthmap_then_landcoverai")
from IPython.display import display, Markdown, Image
import matplotlib.pyplot as plt

assert RESULTS.is_dir(), f"No results directory: {RESULTS}"
for name in ("openearthmap", "landcoverai"):
    folder = RESULTS / name
    display(Markdown(f"### {name}"))
    for filename in ("test_metrics.json", "forgetting.json"):
        path = folder / filename
        if path.is_file():
            print(filename)
            print(json.dumps(json.loads(path.read_text()), indent=2))
        else:
            print("Not available yet:", filename)
    history_path = folder / "history.json"
    if history_path.is_file():
        history = json.loads(history_path.read_text())
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for axis, metric in zip(axes, ("loss", "miou", "dice")):
            for split in ("train", "val"):
                axis.plot([r["epoch"] for r in history],
                          [r[split][metric] for r in history], label=split)
            axis.set(title=metric, xlabel="Epoch")
            axis.legend()
            axis.grid(alpha=0.25)
        fig.suptitle(name)
        fig.tight_layout()
        display(fig)
        plt.close(fig)

## Prediction images and t-SNE

In [ ]:
MAX_IMAGES_PER_TASK = 6
for name in ("openearthmap", "landcoverai"):
    folder = RESULTS / name
    display(Markdown(f"### {name} — predictions"))
    pictures = sorted((folder / "predictions").glob("*.png"))[:MAX_IMAGES_PER_TASK]
    if not pictures:
        print("Prediction images are not available yet.")
    for path in pictures:
        display(Markdown(path.name))
        display(Image(filename=str(path), width=900))
    tsne = folder / "tsne.png"
    if tsne.is_file():
        display(Markdown("t-SNE"))
        display(Image(filename=str(tsne), width=900))

## Save a compact results archive

Includes logs, configs, metrics, and images; excludes large model checkpoints. Retrieve the ZIP using VS Code Remote Explorer's **Download** action. Checkpoints remain on the server at the paths printed below.

In [ ]:
import zipfile

archive = RESULTS.parent / f"{RESULTS.name}_reports.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in RESULTS.parent.rglob("*"):
        if path.is_file() and path.suffix.lower() in {".json", ".csv", ".log", ".yaml", ".png"}:
            bundle.write(path, path.relative_to(RESULTS.parent))
print("Reports archive:", archive)
for checkpoint in sorted(RESULTS.rglob("*.pt")):
    print("Checkpoint:", checkpoint)
print("Save this notebook to keep its displayed outputs.")